# Market Analysis — Emerging RE tertiary/mid-metro picks

Two-approach evaluation for picking top-5 multifamily markets each year:

1. **High-cap rule** (simple heuristic, no training): rank the diverse universe by ACS `cap_rate_proxy` and take the top 5.
2. **Analog-search ML** (walk-forward training): identify historical top-decile winners each cohort, z-score features within cohort, rank candidates by cosine similarity to those winners.

Both use the same universe filter:
- `pop_latest` in target band (default 300K-500K; switchable to 100K-300K)
- `employment_diversity_hhi ≤ 0.10` (well-diversified employment mix)
- Dec-31 cohort dates only

Realized-return metric for both approaches:
- `TR-ZORI36 = (1 + zori_36mo_growth) × (cap_entry / cap_exit_3yr) − 1`
- `cap` = ACS `median_gross_rent × 12 / median_home_value` (SFR+MF blend, PIT-safe)
- Reports cap Δ (bps), realized ZORI rent growth, and composite total return
- ⚠️ **Rent-numerator / cap-numerator mismatch**: ZORI (Zillow monthly, MF-heavy) vs ACS (5yr rolling, blended, includes utilities). Fine for ranking; noisy for absolute TR level.

## 0. Setup

In [13]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '/Users/premputtegowda/reps/reps-tracker/emerging_re')

import duckdb, pandas as pd, numpy as np
from pathlib import Path

WAREHOUSE = "/Users/premputtegowda/reps/reps-tracker/emerging_re/data/warehouse/emerging_re.duckdb"

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

# Universe knobs — switch bands here
POP_MIN, POP_MAX = 300_000, 500_000     # try (100_000, 300_000) for tertiary
DIVERSITY_HHI_MAX = 0.10
HORIZON_YEARS = 3

print(f"Warehouse exists: {Path(WAREHOUSE).exists()}")
print(f"Universe filter: pop {POP_MIN//1000}K-{POP_MAX//1000}K + HHI ≤ {DIVERSITY_HHI_MAX}")

Warehouse exists: True
Universe filter: pop 300K-500K + HHI ≤ 0.1


## 1. Universe overview

Row counts per year in the filtered universe. Sanity check: enough counties per cohort for top-decile winners (≥50 counties/year is comfortable).

In [14]:
con = duckdb.connect(WAREHOUSE, read_only=True)
uni_counts = con.execute(f'''
    SELECT EXTRACT(YEAR FROM period_end_date)::INT AS year,
           COUNT(*) AS n_counties,
           AVG(cap_rate_proxy) AS avg_cap_pct,
           AVG(employment_diversity_hhi) AS avg_hhi
    FROM main_marts.mart_county_features
    WHERE EXTRACT(MONTH FROM period_end_date) = 12
      AND pop_latest > {POP_MIN} AND pop_latest <= {POP_MAX}
      AND employment_diversity_hhi IS NOT NULL
      AND employment_diversity_hhi <= {DIVERSITY_HHI_MAX}
    GROUP BY 1 ORDER BY 1
''').df()
con.close()
uni_counts['avg_cap_pct'] = (uni_counts['avg_cap_pct'] * 100).round(2)
uni_counts['avg_hhi'] = uni_counts['avg_hhi'].round(4)
uni_counts.tail(15)

,year,n_counties,avg_cap_pct,avg_hhi
11,2011,91,5.26,0.0720
12,2012,91,5.50,0.0725
13,2013,91,5.74,0.0734
14,2014,91,5.96,0.0739
15,2015,89,6.03,0.0738
16,2016,89,5.86,0.0740
17,2017,89,5.72,0.0741
18,2018,89,5.63,0.0744
19,2019,89,5.55,0.0739
20,2020,89,5.48,0.0741


## 2. Approach A — High-cap rule (simple heuristic)

**No training.** Just rank by ACS cap-rate proxy each year and take top 5. Fast, transparent, and historically strong when cap rates were compressing.

### 2.1 Load + compute realized returns

In [15]:
def high_cap_load(pop_min, pop_max, hhi_max=DIVERSITY_HHI_MAX):
    """Load universe panel + attach 3yr and 2yr exit caps + compute realized TR."""
    con = duckdb.connect(WAREHOUSE, read_only=True)
    df = con.execute(f'''
        SELECT f.fips, f.county_name, f.state_name, f.pop_latest,
               EXTRACT(YEAR FROM f.period_end_date)::INT AS year,
               f.cap_rate_proxy AS cap_entry,
               f.mf_cap_rate_estimated AS cap_entry_mf,
               f.zori_forward_return_36mo AS zori36,
               f.zori_forward_return_24mo AS zori24,
               f.employment_diversity_hhi AS hhi
        FROM main_marts.mart_county_features f
        WHERE EXTRACT(MONTH FROM f.period_end_date) = 12
          AND f.pop_latest > {pop_min} AND f.pop_latest <= {pop_max}
          AND f.employment_diversity_hhi IS NOT NULL
          AND f.employment_diversity_hhi <= {hhi_max}
    ''').df()
    con.close()
    # Attach exit caps
    for h, col in [(3, 'cap_exit_3yr'), (2, 'cap_exit_2yr')]:
        exit_df = df[['fips','year','cap_entry']].rename(columns={'year': f'ey{h}', 'cap_entry': col})
        df[f'ey{h}'] = df['year'] + h
        df = df.merge(exit_df, on=['fips', f'ey{h}'], how='left')
    df['tr36'] = (1 + df['zori36']) * (df['cap_entry'] / df['cap_exit_3yr']) - 1
    df['tr24'] = (1 + df['zori24']) * (df['cap_entry'] / df['cap_exit_2yr']) - 1
    return df

hc_df = high_cap_load(POP_MIN, POP_MAX)
print(f"Panel: {len(hc_df):,} rows across {hc_df['year'].min()}-{hc_df['year'].max()}")

Panel: 2,309 rows across 2000-2025


### 2.2 Top-5 highest-cap picks per year — with entry cap, exit cap, rent, and total return

In [16]:
def print_high_cap_top5(df, min_year=2015, max_year=2025):
    rows = []
    for yr in sorted(df['year'].unique()):
        if yr < min_year or yr > max_year: continue
        c = df[df['year'] == yr].dropna(subset=['cap_entry'])
        if len(c) < 20: continue
        top5 = c.nlargest(5, 'cap_entry')
        uni_tr = c['tr36'].mean(skipna=True) * 100
        top5_tr = top5['tr36'].mean(skipna=True) * 100
        edge = top5_tr - uni_tr if pd.notna(top5_tr) and pd.notna(uni_tr) else np.nan
        print(f"\n===== {yr} (n={len(c)}) | Top-5 TR {top5_tr:+.2f}% | Universe {uni_tr:+.2f}% | Edge {edge:+.2f}pp =====")
        print(f"{'#':<3}{'Market':<40}{'Pop':>10}   {'Cap in':>8}  {'Cap out':>9}  {'Δ bps':>7}  {'ZORI36':>9}  {'TR':>10}  {'HHI':>7}")
        for i, (_, r) in enumerate(top5.iterrows(), 1):
            name = f"{r['county_name']}, {r['state_name']}"
            pop = f"{int(r['pop_latest']):,}"
            cap_e = f"{r['cap_entry']*100:.2f}%"
            if pd.notna(r['cap_exit_3yr']):
                cap_x = f"{r['cap_exit_3yr']*100:.2f}%"; d = f"{(r['cap_exit_3yr']-r['cap_entry'])*10000:+.0f}"
            elif pd.notna(r['cap_exit_2yr']):
                cap_x = f"{r['cap_exit_2yr']*100:.2f}%*"; d = f"{(r['cap_exit_2yr']-r['cap_entry'])*10000:+.0f}*"
            else:
                cap_x = "n/a"; d = "n/a"
            zori = f"{r['zori36']*100:+.2f}%" if pd.notna(r['zori36']) else (f"{r['zori24']*100:+.2f}%†" if pd.notna(r['zori24']) else "n/a")
            tr = f"{r['tr36']*100:+.2f}%" if pd.notna(r['tr36']) else (f"{r['tr24']*100:+.2f}%†" if pd.notna(r['tr24']) else "n/a")
            hhi = f"{r['hhi']:.4f}"
            print(f"{i:<3}{name:<40}{pop:>10}   {cap_e:>8}  {cap_x:>9}  {d:>7}  {zori:>9}  {tr:>10}  {hhi:>7}")
        rows.append({'year': yr, 'n_universe': len(c), 'top5_TR': top5_tr, 'uni_TR': uni_tr, 'edge_pp': edge,
                     'picks': ' · '.join(top5['county_name'].str.replace(' County','').tolist())})
    print("\n* = 2yr exit used (3yr not observable)   † = 24mo return\n")
    return pd.DataFrame(rows)

hc_summary = print_high_cap_top5(hc_df)


===== 2015 (n=89) | Top-5 TR +32.04% | Universe +20.77% | Edge +11.27pp =====
#  Market                                         Pop     Cap in    Cap out    Δ bps     ZORI36          TR      HHI
1  St. Lucie County, Florida                  402,449      9.84%      8.05%     -179    +19.83%     +46.46%   0.0730
2  Genesee County, Michigan                   401,093      9.76%      7.96%     -179    +12.93%     +38.37%   0.0833
3  Nueces County, Texas                       352,992      9.35%      8.35%     -101     +5.25%     +17.96%   0.0735
4  Marion County, Florida                     442,660      8.81%      7.49%     -132    +16.68%     +37.31%   0.0843
5  Lubbock County, Texas                      328,906      8.81%      7.68%     -113     +4.65%     +20.10%   0.0674

===== 2016 (n=89) | Top-5 TR +21.40% | Universe +19.27% | Edge +2.13pp =====
#  Market                                         Pop     Cap in    Cap out    Δ bps     ZORI36          TR      HHI
1  Nueces County, Texas 

In [17]:
print("=== HIGH-CAP RULE — summary across cohorts ===")
print(hc_summary[['year','n_universe','top5_TR','uni_TR','edge_pp']].to_string(index=False, float_format=lambda x: f'{x:+.2f}'))
obs = hc_summary.dropna(subset=['edge_pp'])
print(f"\nAvg edge: {obs['edge_pp'].mean():+.2f}pp   Won: {(obs['edge_pp']>0).sum()}/{len(obs)} cohorts")

=== HIGH-CAP RULE — summary across cohorts ===
 year  n_universe  top5_TR  uni_TR  edge_pp
 2015          89   +32.04  +20.77   +11.27
 2016          89   +21.40  +19.27    +2.13
 2017          89   +32.72  +19.89   +12.82
 2018          89   +46.18  +29.54   +16.64
 2019          89   +48.60  +40.71    +7.89
 2020          89   +41.28  +43.93    -2.65
 2021          89   +29.80  +26.70    +3.09
 2022          88   +22.36  +14.66    +7.70
 2023          87      NaN     NaN      NaN
 2024          88      NaN     NaN      NaN
 2025          87      NaN     NaN      NaN

Avg edge: +7.36pp   Won: 7/8 cohorts


## 3. Approach B — Analog-search walk-forward training

Reuses the tested `backtest/analog_search.py` module directly (no re-implementation).

Pipeline per cohort year Y:
1. Identify winners = counties in the top 10% of `target_total_return_36mo_demeaned` among cohorts [first_year, Y−1] (need ≥5 train cohorts)
2. Build feature matrix (~60 features) and z-score within each cohort so features are relative to that year's peers
3. For each candidate in year Y, compute cosine similarity to each historical winner in feature space; score = mean of top-5 similarities
4. Pick top 5 with state cap (max 2 per state) — geographic diversification
5. Optional tie-breaker: rerank within top-5 by cap yield + rent momentum + similarity

### 3.1 Load pipeline modules

In [18]:
from backtest.analog_search import (
    load_tertiary_panel, load_mf_context, build_acs_target,
    build_feature_matrix, zscore_within_cohort, walk_forward, print_report,
    TARGET_COL, TARGET_COL_TOTAL_RETURN, HORIZON_YEARS as HZ, MIN_TRAIN_YEARS,
)
print(f"Imported. Target column options: rent={TARGET_COL}, total_return={TARGET_COL_TOTAL_RETURN}")
print(f"Horizon: {HZ} years   Min train cohorts: {MIN_TRAIN_YEARS}")

Imported. Target column options: rent=target_acs_rent_36mo_demeaned, total_return=target_total_return_36mo_demeaned
Horizon: 3 years   Min train cohorts: 5


### 3.2 Build the training data

`load_tertiary_panel()` accepts `pop_min`/`pop_max`/`hhi_max` as of the recent parameterization — the cell below threads the notebook's `POP_MIN`/`POP_MAX`/`DIVERSITY_HHI_MAX` knobs through, so the ML pipeline uses the same universe as the high-cap section.


In [19]:
# Universe panel + realized target (rent + cap + composite total return)
panel = load_tertiary_panel(pop_min=POP_MIN, pop_max=POP_MAX, hhi_max=DIVERSITY_HHI_MAX)
print(f"Panel: {len(panel):,} rows across years {panel['year'].min()}-{panel['year'].max()}")

acs_target = build_acs_target(horizon_years=HZ)
print(f"ACS target: {len(acs_target):,} rows, {acs_target[TARGET_COL].notna().sum():,} with observed 36mo rent target")

panel = panel.merge(acs_target, on=["fips", "period_end_date"], how="left")
print(f"After target merge: {panel[TARGET_COL].notna().sum():,} panel rows have observed 36mo target")

# MF context (unit counts, building estimates) — for interpretability display, not training
mf_ctx = load_mf_context()
panel = panel.merge(mf_ctx, on="fips", how="left")
print(f"MF context attached to {panel['mf_units_5plus'].notna().sum():,} rows")

Panel: 2,309 rows across years 2000-2025
ACS target: 39,358 rows, 39,358 with observed 36mo rent target
After target merge: 1,173 panel rows have observed 36mo target
MF context attached to 2,309 rows


### 3.3 Build feature matrix + z-score within cohort

In [20]:
panel_fe, feature_cols = build_feature_matrix(panel)
print(f"Feature matrix: {len(panel_fe):,} rows × {len(feature_cols)} features")
print(f"\nSample features (first 15): {feature_cols[:15]}")

panel_z = zscore_within_cohort(panel_fe, feature_cols)
print(f"\nZ-scored panel ready.")

Feature matrix: 2,309 rows × 91 features

Sample features (first 15): ['domesticmig_per_1000', 'domesticmig_accel_3yr', 'high_wage_share', 'high_wage_share_yoy_chg', 'wage_growth_yoy', 'wage_growth_yoy_demeaned', 'emp_growth_yoy', 'emp_growth_yoy_demeaned', 'permits_per_1000', 'permits_per_1000_10yr_baseline', 'permits_intensity_ratio', 'permits_10yr_baseline_years_available', 'hpi_yoy', 'zori_yoy', 'rent_minus_price_growth']

Z-scored panel ready.


### 3.4 Run walk-forward training

Uses **composite total-return target** (rent × cap compression) — the primary target as of the recent changes.

Prints per-cohort schedule + verbose per-year picks.

In [21]:
results = walk_forward(
    panel_z, panel_fe, feature_cols,
    test_years=None,                       # None = auto-generate expanding schedule
    min_train_years=MIN_TRAIN_YEARS,
    target_col=TARGET_COL_TOTAL_RETURN,    # composite (rent × cap compression) demeaned
)
print(f"\nCompleted {len(results)} walk-forward iterations")


Walk-forward schedule (21 iterations, min_train_years=5, target=target_total_return_36mo_demeaned):
  SKIP test=2005   train_cohorts= 0   span=(none)
  SKIP test=2006   train_cohorts= 0   span=(none)
  SKIP test=2007   train_cohorts= 0   span=(none)
  SKIP test=2008   train_cohorts= 0   span=(none)
  SKIP test=2009   train_cohorts= 1   span=2008-2008
  SKIP test=2010   train_cohorts= 2   span=2008-2009
  SKIP test=2011   train_cohorts= 3   span=2008-2010
  SKIP test=2012   train_cohorts= 4   span=2008-2011
  test=2013   train_cohorts= 5   span=2008-2012
  test=2014   train_cohorts= 6   span=2008-2013
  test=2015   train_cohorts= 7   span=2008-2014
  test=2016   train_cohorts= 8   span=2008-2015
  test=2017   train_cohorts= 9   span=2008-2016
  test=2018   train_cohorts=10   span=2008-2017
  test=2019   train_cohorts=11   span=2008-2018
  test=2020   train_cohorts=12   span=2008-2019
  test=2021   train_cohorts=12   span=2008-2019
  test=2022   train_cohorts=13   span=2008-2021
  test=

### 3.5 Full report — per cohort picks + realized returns + tie-breaker + winner analogs

In [22]:
print_report(results, panel_raw=panel_fe)


ANALOG-SEARCH TOP-5 — feature similarity to historical top-decile winners
Universe: pop 300K-491K + emp_diversity_hhi <= 0.1   Training: 2008-2018 cohorts   Winner def: top 10% by ACS 36mo demeaned
Similarity: cosine on z-scored features (within cohort). Score = mean of top-5 matches.
Eval: report BOTH ACS 36mo (observable 2019 only) AND ZORI 36mo (observable 2015+ through 2022).


===== TEST YEAR 2013  (train cohorts=5, winners=50) =====
  ACS RENT 36mo:  top5= +13.10% (  +4.18%/yr)   uni= +10.65% (  +3.42%/yr)   edge=  +2.46pp (  +0.76pp/yr)
  TOTAL RETN 36mo:top5= +20.29% (  +6.31%/yr)   uni= +12.47% (  +3.93%/yr)   edge=  +7.82pp (  +2.38pp/yr)
  ZORI 36mo:      top5=   n/a   (   n/a  )   uni=   n/a   (   n/a  )   edge=   n/a   (   n/a  )
  ZORI 24mo:      top5=   n/a   (   n/a  )   uni=   n/a   (   n/a  )   edge=   n/a   (   n/a  )
  #1  [TB #3, score=0.680]  Weld County            Colorado         pop=378,426   sim=0.7434   cap 5.16%→4.52% ( -65bps)   ACS36=+12.17% (+3.90%/yr)  

### 3.6 Extract structured per-year summary from walk-forward results

In [23]:
ml_rows = []
for r in results:
    top5 = r.top5
    def _mean(col):
        return top5[col].mean(skipna=True) * 100 if col in top5.columns else np.nan
    ml_rows.append({
        'year': r.test_year,
        'n_train_cohorts': r.n_train_cohorts,
        'n_winners': r.n_winners,
        'top5_TR_zori36': _mean('total_return_zori36_36mo'),
        'uni_TR_zori36':  r.universe_mean_zori_36mo * 100 if not np.isnan(r.universe_mean_zori_36mo) else np.nan,  # note: uses zori36 rent, cap held const
        'top5_ZORI36':    _mean('zori_forward_return_36mo'),
        'top5_ACS36':     _mean('acs_forward_return_36mo'),
        'picks': ' · '.join(r.top5['county_name'].str.replace(' County','').tolist()),
    })
ml_summary = pd.DataFrame(ml_rows)
ml_summary

,year,n_train_cohorts,n_winners,top5_TR_zori36,uni_TR_zori36,top5_ZORI36,top5_ACS36,picks
0,2013,5,50,NaN,NaN,NaN,13.100133,Weld · Boulder · Benton · Collier · Cleveland
1,2014,6,60,NaN,NaN,NaN,16.492449,Weld · Boulder · Collier · Sarasota · Lexington
2,2015,7,70,29.755981,12.066302,12.793970,16.561812,Boulder · Larimer · Benton · Seminole · Sarasota
3,2016,8,79,30.850176,12.571833,14.856382,13.655218,Lake · St. Lucie · Boulder · Larimer · Benton
4,2017,9,88,34.241212,14.510385,14.185551,21.437928,St. Lucie · Boulder · Weld · Lake · Rutherford
5,2018,10,97,43.516240,25.597696,30.454751,17.070116,Boulder · St. Lucie · Lake · York · Rutherford
6,2019,11,106,49.080595,29.185175,32.691272,20.650672,Lake · Rutherford · St. Lucie · Davis · Weld
7,2020,12,115,57.769329,26.416656,31.559329,NaN,Davis · Madison · Cumberland · St. Lucie · Lake
8,2021,12,115,30.212876,14.309394,9.640482,32.300626,St. Lucie · Osceola · Cumberland · Davis · Hays
9,2022,13,124,7.414060,9.643768,4.027098,NaN,Davis · Lake · Seminole · Charleston · Hays


## 4. Head-to-head — high-cap rule vs analog-search ML

Same universe, same horizon, same total-return metric. Which approach won each cohort?

In [24]:
# Align both summaries on year
hc_ = hc_summary[['year','top5_TR','uni_TR','edge_pp','picks']].rename(
    columns={'top5_TR':'hc_top5_TR', 'edge_pp':'hc_edge_pp', 'picks':'hc_picks'})
ml_ = ml_summary[['year','top5_TR_zori36','picks']].rename(
    columns={'top5_TR_zori36':'ml_top5_TR', 'picks':'ml_picks'})

cmp = hc_.merge(ml_, on='year', how='outer').sort_values('year')
cmp['ml_edge_pp'] = cmp['ml_top5_TR'] - cmp['uni_TR']
cmp['ml_vs_hc_pp'] = cmp['ml_top5_TR'] - cmp['hc_top5_TR']

print("=== HIGH-CAP vs ANALOG-SEARCH — realized TR per cohort ===")
print(cmp[['year','uni_TR','hc_top5_TR','ml_top5_TR','hc_edge_pp','ml_edge_pp','ml_vs_hc_pp']].to_string(index=False, float_format=lambda x: f'{x:+.2f}'))

obs = cmp.dropna(subset=['ml_vs_hc_pp'])
print(f"\nAvg high-cap edge vs universe:      {obs['hc_edge_pp'].mean():+.2f}pp")
print(f"Avg analog-search edge vs universe: {obs['ml_edge_pp'].mean():+.2f}pp")
print(f"Avg ML lift over high-cap rule:     {obs['ml_vs_hc_pp'].mean():+.2f}pp   (ML won {(obs['ml_vs_hc_pp']>0).sum()}/{len(obs)} cohorts)")

=== HIGH-CAP vs ANALOG-SEARCH — realized TR per cohort ===
 year  uni_TR  hc_top5_TR  ml_top5_TR  hc_edge_pp  ml_edge_pp  ml_vs_hc_pp
 2013     NaN         NaN         NaN         NaN         NaN          NaN
 2014     NaN         NaN         NaN         NaN         NaN          NaN
 2015  +20.77      +32.04      +29.76      +11.27       +8.99        -2.28
 2016  +19.27      +21.40      +30.85       +2.13      +11.58        +9.45
 2017  +19.89      +32.72      +34.24      +12.82      +14.35        +1.52
 2018  +29.54      +46.18      +43.52      +16.64      +13.98        -2.66
 2019  +40.71      +48.60      +49.08       +7.89       +8.37        +0.48
 2020  +43.93      +41.28      +57.77       -2.65      +13.84       +16.49
 2021  +26.70      +29.80      +30.21       +3.09       +3.51        +0.42
 2022  +14.66      +22.36       +7.41       +7.70       -7.25       -14.95
 2023     NaN         NaN         NaN         NaN         NaN          NaN
 2024     NaN         NaN         NaN    

### Side-by-side picks per year

In [25]:
for _, r in cmp.iterrows():
    print(f"\n{int(r['year'])}:  HC top5 = {r['hc_picks']}")
    if isinstance(r['ml_picks'], str):
        print(f"      ML top5 = {r['ml_picks']}")


2013:  HC top5 = nan
      ML top5 = Weld · Boulder · Benton · Collier · Cleveland

2014:  HC top5 = nan
      ML top5 = Weld · Boulder · Collier · Sarasota · Lexington

2015:  HC top5 = St. Lucie · Genesee · Nueces · Marion · Lubbock
      ML top5 = Boulder · Larimer · Benton · Seminole · Sarasota

2016:  HC top5 = Nueces · Genesee · Escambia · Cumberland · Lubbock
      ML top5 = Lake · St. Lucie · Boulder · Larimer · Benton

2017:  HC top5 = Nueces · Escambia · St. Lucie · Lubbock · Genesee
      ML top5 = St. Lucie · Boulder · Weld · Lake · Rutherford

2018:  HC top5 = Nueces · St. Lucie · Genesee · Escambia · Lubbock
      ML top5 = Boulder · St. Lucie · Lake · York · Rutherford

2019:  HC top5 = Nueces · Cumberland · Escambia · Lucas · Genesee
      ML top5 = Lake · Rutherford · St. Lucie · Davis · Weld

2020:  HC top5 = Cumberland · Nueces · Luzerne · Lubbock · Lucas
      ML top5 = Davis · Madison · Cumberland · St. Lucie · Lake

2021:  HC top5 = Cumberland · Nueces · Luzerne 

## 5. Today's picks (2025 cohort) — both approaches

In [26]:
# High-cap 2025 picks
c25 = hc_df[hc_df['year'] == 2025].dropna(subset=['cap_entry']).nlargest(5, 'cap_entry')
print(f"=== HIGH-CAP RULE — 2025 picks ({POP_MIN//1000}-{POP_MAX//1000}K + diverse) ===")
for i, (_, r) in enumerate(c25.iterrows(), 1):
    mf_str = f"{r['cap_entry_mf']*100:.2f}%" if pd.notna(r['cap_entry_mf']) else "n/a"
    print(f"  #{i}  {r['county_name']}, {r['state_name']:20s}  pop={int(r['pop_latest']):>7,}   "
          f"cap_ACS={r['cap_entry']*100:5.2f}%   cap_MF={mf_str}   HHI={r['hhi']:.4f}")

# ML 2025 picks (from most recent walk-forward result at year 2025)
print(f"\n=== ANALOG-SEARCH ML — 2025 picks (same universe) ===")
ml_2025 = next((r for r in results if r.test_year == 2025), None)
if ml_2025 is not None:
    for i, (_, r) in enumerate(ml_2025.top5.iterrows(), 1):
        cap_e = r.get('cap_rate_proxy', float('nan')) * 100
        print(f"  #{i}  {r['county_name']}, {r['state_name']:20s}  pop={int(r['pop_latest']):>7,}   "
              f"cap={cap_e:5.2f}%   sim={r['analog_score']:.4f}")
else:
    print("  (2025 iteration not in results — check walk-forward schedule)")

=== HIGH-CAP RULE — 2025 picks (300-500K + diverse) ===
  #1  Cumberland County, North Carolina        pop=338,473   cap_ACS= 7.24%   cap_MF=8.22%   HHI=0.0563
  #2  Marion County, Florida               pop=442,660   cap_ACS= 6.63%   cap_MF=7.48%   HHI=0.0836
  #3  Nueces County, Texas                 pop=352,992   cap_ACS= 6.60%   cap_MF=8.60%   HHI=0.0819
  #4  Luzerne County, Pennsylvania          pop=332,126   cap_ACS= 6.55%   cap_MF=8.96%   HHI=0.0864
  #5  Lubbock County, Texas                 pop=328,906   cap_ACS= 6.52%   cap_MF=7.72%   HHI=0.0671

=== ANALOG-SEARCH ML — 2025 picks (same universe) ===
  #1  Benton County, Arkansas              pop=332,554   cap=-72.26%   sim=0.6867
  #2  Henrico County, Virginia              pop=342,775   cap=35.98%   sim=0.6364
  #3  Albany County, New York              pop=321,225   cap=36.11%   sim=0.6020
  #4  Chesterfield County, Virginia              pop=397,148   cap=36.06%   sim=0.5667
  #5  Hays County, Texas                 pop=304,39

## 6. Diagnostic — returns by cap-rate quintile

Is the pattern monotonic? Higher cap → higher realized TR across the whole distribution?

In [27]:
obs = hc_df.dropna(subset=['cap_entry','zori36','cap_exit_3yr']).copy()
obs = obs[(obs['year'] >= 2015) & (obs['year'] <= 2022)]

qrows = []
for y in sorted(obs['year'].unique()):
    c = obs[obs['year'] == y].copy()
    c['quintile'] = pd.qcut(c['cap_entry'], q=5, labels=['Q1','Q2','Q3','Q4','Q5'], duplicates='drop')
    for q in ['Q1','Q2','Q3','Q4','Q5']:
        sub = c[c['quintile'] == q]
        if len(sub):
            qrows.append({'year': y, 'quintile': q, 'n': len(sub),
                'avg_cap_pct': sub['cap_entry'].mean()*100,
                'avg_zori36_pct': sub['zori36'].mean()*100,
                'cap_delta_bps': (sub['cap_exit_3yr'].mean() - sub['cap_entry'].mean())*10000,
                'avg_TR_pct': sub['tr36'].mean()*100})
q = pd.DataFrame(qrows)

print("=== Realized TR-ZORI36 by cap quintile per year (%) ===")
print(q.pivot(index='year', columns='quintile', values='avg_TR_pct').round(2).to_string())

print("\n=== SUMMARY: avg per quintile across all observable cohorts ===")
print(q.groupby('quintile', observed=True).agg(
    avg_cap_pct=('avg_cap_pct','mean'),
    avg_cap_delta_bps=('cap_delta_bps','mean'),
    avg_zori36_pct=('avg_zori36_pct','mean'),
    avg_TR_pct=('avg_TR_pct','mean')).round(2).to_string())

=== Realized TR-ZORI36 by cap quintile per year (%) ===
quintile     Q1     Q2     Q3     Q4     Q5
year                                       
2015      21.93  17.76  15.78  22.23  25.84
2016      18.73  22.28  13.74  18.24  23.45
2017      14.67  19.25  21.54  17.98  25.98
2018      24.40  27.02  31.15  29.95  35.05
2019      32.88  35.13  46.01  39.13  50.00
2020      38.95  40.47  44.28  48.05  47.96
2021      22.78  23.41  25.84  29.40  32.06
2022      11.83   9.18  12.97  19.97  19.26

=== SUMMARY: avg per quintile across all observable cohorts ===
          avg_cap_pct  avg_cap_delta_bps  avg_zori36_pct  avg_TR_pct
quintile                                                            
Q1               4.04             -19.73           16.81       23.27
Q2               4.97             -23.35           17.78       24.31
Q3               5.55             -33.59           17.76       26.41
Q4               6.14             -42.87           18.54       28.12
Q5               7.33    

## 7. Caveats — what these numbers do and don't mean

**Directional, not literal.** The absolute total-return numbers (e.g., "+46%") mix measurement sources:
- Rent numerator: ZORI (Zillow, monthly, MF-heavy inventory)
- Cap denominator: `cap_rate_proxy` = ACS `median_gross_rent × 12 / median_home_value` (all renters, includes utilities, ACS 5yr rolling)

These are different rent measures pulling on the same identity `V = NOI / cap`. Ranking is preserved across markets in a cohort; absolute levels have ~5-10pp of measurement noise.

**Cap compression tailwind.** The 2015-2021 edge relied on falling rates → capital chasing yield → cap rates compressing. In a rising-rate regime (2022+), compression stalls or reverses. Q5 lost to Q4 slightly in 2022, and the 2023 cohort's short-horizon returns look negative for Sunbelt picks.

**High-cap ≠ safe.** The strategy captures markets where the market demands a high yield — often for good reasons (population loss, single-anchor risk, ageing stock). The `HHI ≤ 0.10` and `disq_*` gates screen out the worst, but markets like Flint (Genesee MI) and Toledo (Lucas OH) carry structural challenges even inside the filter.

**MF cap rate is a proxy.** Real MF acquisition caps depend on class (A/B/C), age, unit mix — none captured here. Use `mf_cap_rate_estimated` (HUD FMR-based + 1.5% MF premium) as a directional MF adjustment.

**Short horizons amplify noise.** 24mo returns (2023 cohort) are dominated by endpoint cap-rate measurement noise. Prefer 36mo when observable.

**Universe filter matters.** Small populations (100-300K band) show larger edges (+11.6pp) than mid-metros (300-500K band, +7.4pp). Smaller markets are less efficient. But they're also less liquid — real acquisition of even one property in Robeson NC or Potter TX is harder than in Nueces or Cumberland.

## 8. A/B test — HUD-derived cap (`mf_cap_rate_estimated`) vs ACS-derived cap (`cap_rate_proxy`)

**Question:** Would using the MF-adjusted HUD FMR cap materially change picks and edges vs the ACS blended cap we currently use?

**Approach:** Same universe + horizon. Two identical pipelines, only the cap column differs:
- **ACS cap:** `cap_entry = cap_exit = cap_rate_proxy` (ACS median gross rent × 12 / median home value; SFR+MF blend, includes utilities)
- **HUD cap:** `cap_entry = cap_exit = mf_cap_rate_estimated` (HUD FMR 2BR × 12 / median home value + 1.5% MF premium)

**Metrics compared:**
1. **Overlap** — of the 5 picks per year, how many are the same?
2. **Edge shift** — does the +7pp/+11pp historical edge change materially?
3. **Composition** — do FL/TX Sunbelt vs Rust Belt tilts change?

In [28]:
def high_cap_load_dual(pop_min, pop_max, hhi_max=DIVERSITY_HHI_MAX):
    """Load panel with BOTH cap sources; compute TR two ways."""
    con = duckdb.connect(WAREHOUSE, read_only=True)
    df = con.execute(f'''
        SELECT f.fips, f.county_name, f.state_name, f.pop_latest,
               EXTRACT(YEAR FROM f.period_end_date)::INT AS year,
               f.cap_rate_proxy AS cap_acs,
               f.mf_cap_rate_estimated AS cap_hud,
               f.zori_forward_return_36mo AS zori36,
               f.employment_diversity_hhi AS hhi
        FROM main_marts.mart_county_features f
        WHERE EXTRACT(MONTH FROM f.period_end_date) = 12
          AND f.pop_latest > {pop_min} AND f.pop_latest <= {pop_max}
          AND f.employment_diversity_hhi IS NOT NULL
          AND f.employment_diversity_hhi <= {hhi_max}
    ''').df()
    con.close()

    for cap_col, out_col in [('cap_acs', 'cap_acs_exit'), ('cap_hud', 'cap_hud_exit')]:
        exit_df = df[['fips','year',cap_col]].rename(columns={'year':'ey', cap_col: out_col})
        df['ey'] = df['year'] + 3
        df = df.merge(exit_df, on=['fips','ey'], how='left').drop(columns=['ey'])

    df['tr_acs'] = (1 + df['zori36']) * (df['cap_acs'] / df['cap_acs_exit']) - 1
    df['tr_hud'] = (1 + df['zori36']) * (df['cap_hud'] / df['cap_hud_exit']) - 1
    return df

ab_df = high_cap_load_dual(POP_MIN, POP_MAX)
print(f"Panel: {len(ab_df):,} rows across {ab_df['year'].min()}-{ab_df['year'].max()}")
print(f"Rows with both cap columns available at entry:")
print(f"  cap_acs: {ab_df['cap_acs'].notna().sum():,}")
print(f"  cap_hud: {ab_df['cap_hud'].notna().sum():,}")
print(f"  both:    {(ab_df['cap_acs'].notna() & ab_df['cap_hud'].notna()).sum():,}")

Panel: 2,309 rows across 2000-2025
Rows with both cap columns available at entry:
  cap_acs: 1,337
  cap_hud: 706
  both:    706


### 8.1 Side-by-side top-5 per year — ACS-ranked vs HUD-ranked

In [29]:
def top5(df, cap_col, yr):
    c = df[df['year'] == yr].dropna(subset=[cap_col])
    if len(c) < 20: return None
    return c.nlargest(5, cap_col)

ab_rows = []
for yr in sorted(ab_df['year'].unique()):
    if yr < 2015 or yr > 2025: continue
    t_acs = top5(ab_df, 'cap_acs', yr)
    t_hud = top5(ab_df, 'cap_hud', yr)
    if t_acs is None or t_hud is None: continue

    acs_fips = set(t_acs['fips'])
    hud_fips = set(t_hud['fips'])
    overlap = len(acs_fips & hud_fips)

    n_uni = len(ab_df[ab_df['year'] == yr])
    tr_acs_top5 = t_acs['tr_acs'].mean(skipna=True) * 100
    tr_hud_top5 = t_hud['tr_hud'].mean(skipna=True) * 100
    tr_uni_acs  = ab_df[ab_df['year'] == yr]['tr_acs'].mean(skipna=True) * 100
    tr_uni_hud  = ab_df[ab_df['year'] == yr]['tr_hud'].mean(skipna=True) * 100

    ab_rows.append({
        'year': yr,
        'n_uni': n_uni,
        'overlap_5': overlap,
        'ACS_top5_TR': tr_acs_top5,
        'ACS_uni_TR':  tr_uni_acs,
        'ACS_edge':    tr_acs_top5 - tr_uni_acs if pd.notna(tr_acs_top5) and pd.notna(tr_uni_acs) else np.nan,
        'HUD_top5_TR': tr_hud_top5,
        'HUD_uni_TR':  tr_uni_hud,
        'HUD_edge':    tr_hud_top5 - tr_uni_hud if pd.notna(tr_hud_top5) and pd.notna(tr_uni_hud) else np.nan,
    })

    # Print picks side by side
    print(f"\n===== {yr} (universe n={n_uni}, overlap: {overlap}/5) =====")
    print(f"  {'#':<3} {'ACS-ranked pick':<32} {'cap_acs':>8}   {'HUD-ranked pick':<32} {'cap_hud':>8}")
    for i in range(5):
        a_row = t_acs.iloc[i]
        h_row = t_hud.iloc[i]
        a_name = f"{a_row['county_name']}, {a_row['state_name'][:2]}"
        h_name = f"{h_row['county_name']}, {h_row['state_name'][:2]}"
        same = '=' if a_row['fips'] == h_row['fips'] else ' '
        print(f"  {i+1:<3} {a_name:<32} {a_row['cap_acs']*100:>7.2f}% {same} {h_name:<32} {h_row['cap_hud']*100:>7.2f}%")

ab_summary = pd.DataFrame(ab_rows)
print()


===== 2018 (universe n=89, overlap: 3/5) =====
  #   ACS-ranked pick                   cap_acs   HUD-ranked pick                   cap_hud
  1   Nueces County, Te                   8.35% = Nueces County, Te                  11.11%
  2   St. Lucie County, Fl                8.05%   Genesee County, Mi                  9.71%
  3   Genesee County, Mi                  7.96%   Escambia County, Fl                 9.46%
  4   Escambia County, Fl                 7.95%   Cumberland County, No               9.35%
  5   Lubbock County, Te                  7.68%   Mobile County, Al                   9.26%

===== 2019 (universe n=89, overlap: 4/5) =====
  #   ACS-ranked pick                   cap_acs   HUD-ranked pick                   cap_hud
  1   Nueces County, Te                   8.56% = Nueces County, Te                  10.00%
  2   Cumberland County, No               8.01%   Mobile County, Al                   9.46%
  3   Escambia County, Fl                 7.68%   Genesee County, Mi        

### 8.2 Edge comparison — does the choice of cap column change the historical edge?

In [30]:
print("=== ACS-cap vs HUD-cap — realized 36mo TR edge over universe ===")
print(ab_summary[['year','n_uni','overlap_5','ACS_top5_TR','HUD_top5_TR','ACS_edge','HUD_edge']].to_string(index=False, float_format=lambda x: f'{x:+.2f}'))

obs = ab_summary.dropna(subset=['ACS_edge','HUD_edge'])
print(f"\n--- SUMMARY over {len(obs)} observable cohorts ---")
print(f"Avg ACS-cap edge:        {obs['ACS_edge'].mean():+.2f}pp   won {(obs['ACS_edge']>0).sum()}/{len(obs)} cohorts")
print(f"Avg HUD-cap edge:        {obs['HUD_edge'].mean():+.2f}pp   won {(obs['HUD_edge']>0).sum()}/{len(obs)} cohorts")
print(f"Avg lift (HUD - ACS):    {(obs['HUD_edge'] - obs['ACS_edge']).mean():+.2f}pp   (HUD better in {(obs['HUD_edge']>obs['ACS_edge']).sum()}/{len(obs)} cohorts)")

print(f"\n--- Pick overlap ---")
print(f"Avg overlap (of 5 picks): {ab_summary['overlap_5'].mean():.1f} / 5")
print(f"Cohorts with ≥4 overlap:  {(ab_summary['overlap_5'] >= 4).sum()} / {len(ab_summary)}")
print(f"Cohorts with ≤2 overlap:  {(ab_summary['overlap_5'] <= 2).sum()} / {len(ab_summary)}")

=== ACS-cap vs HUD-cap — realized 36mo TR edge over universe ===
 year  n_uni  overlap_5  ACS_top5_TR  HUD_top5_TR  ACS_edge  HUD_edge
 2018     89          3       +46.18       +39.66    +16.64    +13.60
 2019     89          4       +48.60       +36.36     +7.89     +7.64
 2020     89          4       +41.28       +24.94     -2.65     -2.50
 2021     89          4       +29.80       +26.26     +3.09     +6.11
 2022     88          4       +22.36       +24.88     +7.70     +9.16
 2023     87          5          NaN          NaN       NaN       NaN
 2024     88          3          NaN          NaN       NaN       NaN
 2025     87          2          NaN          NaN       NaN       NaN

--- SUMMARY over 5 observable cohorts ---
Avg ACS-cap edge:        +6.54pp   won 4/5 cohorts
Avg HUD-cap edge:        +6.80pp   won 4/5 cohorts
Avg lift (HUD - ACS):    +0.26pp   (HUD better in 3/5 cohorts)

--- Pick overlap ---
Avg overlap (of 5 picks): 3.6 / 5
Cohorts with ≥4 overlap:  5 / 8
Cohorts w

### 8.3 Interpretation

Read the summary above:
- **If overlap is ≥4/5 on average**: the two caps rank markets almost identically. The choice is cosmetic — stick with ACS (broader coverage, PIT-safe, established).
- **If overlap is ≤2/5**: the caps disagree materially. Which one wins on realized edge should dictate the choice.
- **If HUD edge is meaningfully higher (>+2pp)**: switch to HUD as the primary cap; the ACS blending was contaminating the signal.
- **If HUD edge is meaningfully lower**: the extra "MF-adjustment" of HUD-cap is noise, and the ACS blend actually captures fundamentals better in aggregate.

**Caveat to keep in mind:** In both TR calculations we still use ZORI as the rent-growth numerator. Neither approach fully resolves the rent-source mismatch. A true apples-to-apples MF cap+rent identity would require FMR-based rent growth too — but FMR moves slowly on HUD's annual cycle and would degrade the freshness of the total-return signal.